## Q1. Spark Initialization and Data Loading

In [7]:
!pip install findspark

In [4]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("ManufacturingDataAnalysis").getOrCreate()

print("Spark Session Initialized.")

Spark Session Initialized.


In [5]:
# Load manufacturing datasets
quality_control_df = spark.read.csv('/content/quality_control.csv', header=True, inferSchema=True)
maintenance_records_df = spark.read.csv('/content/maintenance_records.csv', header=True, inferSchema=True)
machines_df = spark.read.csv('/content/machines.csv', header=True, inferSchema=True)
sensor_data_df = spark.read.csv('/content/sensor_data.csv', header=True, inferSchema=True)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [8]:
print("Quality Control Data:")
quality_control_df.show(5)
quality_control_df.printSchema()

print("\nMaintenance Records Data:")
maintenance_records_df.show(5)
maintenance_records_df.printSchema()

print("\nMachines Data:")
machines_df.show(5)
machines_df.printSchema()

print("\nSensor Data:")
sensor_data_df.show(5)
sensor_data_df.printSchema()

Quality Control Data:
+-------+----------+----------+---------------+------------+--------------+---------+
|  qc_id|machine_id|  batch_id|inspection_date|defect_count|units_produced|pass_fail|
+-------+----------+----------+---------------+------------+--------------+---------+
|QC00001|      M001|BATCH00001|     2025-03-27|          15|           594|     PASS|
|QC00002|      M001|BATCH00002|     2025-02-08|          16|           978|     PASS|
|QC00003|      M001|BATCH00003|     2025-01-01|           2|           114|     PASS|
|QC00004|      M001|BATCH00004|     2025-03-26|           3|           273|     PASS|
|QC00005|      M001|BATCH00005|     2025-02-10|          20|           981|     PASS|
+-------+----------+----------+---------------+------------+--------------+---------+
only showing top 5 rows
root
 |-- qc_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- inspection_date: date (nullable = true)
 |-- defect

## Q2. RDD Implementation


In [9]:
# Convert DataFrames to RDDs
quality_control_rdd = quality_control_df.rdd
maintenance_records_rdd = maintenance_records_df.rdd
machines_rdd = machines_df.rdd
sensor_data_rdd = sensor_data_df.rdd

print("DataFrames successfully converted to RDDs.")

DataFrames successfully converted to RDDs.


### RDD Transformation

Let's perform a transformation on `quality_control_rdd` to select `machine_id`, `defect_count`, and `pass_fail` columns, and then filter for records where `pass_fail` is 'FAIL'. Finally, we'll use an action to collect and display the first 5 results.

In [11]:
failed_qc_rdd = quality_control_rdd \
    .map(lambda row: (row['machine_id'], row['defect_count'], row['pass_fail'])) \
    .filter(lambda record: record[2] == 'FAIL')

# Action: Collect and display the first 5 results
print("First 5 records of failed quality control (Machine ID, Defect Count, Pass/Fail):")
for record in failed_qc_rdd.take(5):
    print(record)

First 5 records of failed quality control (Machine ID, Defect Count, Pass/Fail):
('M015', 59, 'FAIL')
('M020', 29, 'FAIL')


## Q3. Key-Value Operations and Persistence


In [12]:
# Create a key-value RDD (machine_id, defect_count)
machine_defect_rdd = quality_control_rdd.map(lambda row: (row['machine_id'], row['defect_count']))

# Perform reduceByKey to sum defect counts per machine
total_defects_per_machine = machine_defect_rdd.reduceByKey(lambda a, b: a + b)

print("Total defect counts per machine (first 5 results):")
for machine, total_defects in total_defects_per_machine.take(5):
    print(f"Machine ID: {machine}, Total Defects: {total_defects}")

Total defect counts per machine (first 5 results):
Machine ID: M001, Total Defects: 400
Machine ID: M002, Total Defects: 462
Machine ID: M003, Total Defects: 607
Machine ID: M004, Total Defects: 364
Machine ID: M005, Total Defects: 583


### Persistence Techniques



In [13]:
from pyspark import StorageLevel

# Cache the machine_defect_rdd
machine_defect_rdd.cache()
print(f"Is machine_defect_rdd cached? {machine_defect_rdd.is_cached}")

# Perform an action to trigger caching (if not already triggered by previous actions)
_ = machine_defect_rdd.count()
print(f"Is machine_defect_rdd cached after action? {machine_defect_rdd.is_cached}")

# Unpersist/Uncache the RDD
machine_defect_rdd.unpersist()
print(f"Is machine_defect_rdd cached after unpersisting? {machine_defect_rdd.is_cached}")

Is machine_defect_rdd cached? True
Is machine_defect_rdd cached after action? True
Is machine_defect_rdd cached after unpersisting? False


## Q4. Spark DataFrame Operations



In [14]:
# Join quality_control_df with machines_df
quality_control_with_machine_details_df = quality_control_df.join(
    machines_df,
    on='machine_id',
    how='inner'
)

print("Joined DataFrame (first 5 rows):")
quality_control_with_machine_details_df.show(5)
quality_control_with_machine_details_df.printSchema()

Joined DataFrame (first 5 rows):
+----------+-------+----------+---------------+------------+--------------+---------+----------------+------------+--------------+-----------------------+------------+
|machine_id|  qc_id|  batch_id|inspection_date|defect_count|units_produced|pass_fail|    machine_type|install_date|      location|capacity_units_per_hour|manufacturer|
+----------+-------+----------+---------------+------------+--------------+---------+----------------+------------+--------------+-----------------------+------------+
|      M001|QC00001|BATCH00001|     2025-03-27|          15|           594|     PASS|Injection_Molder|  2021-03-30|Plant_A_Floor2|                    437|     Siemens|
|      M001|QC00002|BATCH00002|     2025-02-08|          16|           978|     PASS|Injection_Molder|  2021-03-30|Plant_A_Floor2|                    437|     Siemens|
|      M001|QC00003|BATCH00003|     2025-01-01|           2|           114|     PASS|Injection_Molder|  2021-03-30|Plant_A_Floo

### Aggregation Analysis



In [15]:
from pyspark.sql.functions import avg

# Calculate average defect count per machine type
defects_by_machine_type_df = quality_control_with_machine_details_df \
    .groupBy('machine_type') \
    .agg(avg('defect_count').alias('average_defect_count')) \
    .orderBy('average_defect_count', ascending=False)

print("Average defect count per machine type:")
defects_by_machine_type_df.show()

Average defect count per machine type:
+----------------+--------------------+
|    machine_type|average_defect_count|
+----------------+--------------------+
|     Robotic_Arm|  17.676923076923078|
| Hydraulic_Press|  17.413333333333334|
|Injection_Molder|  16.211382113821138|
|   Conveyor_Belt|  15.937062937062937|
|   Welding_Robot|   14.03076923076923|
+----------------+--------------------+



## Q5. Exploratory Data Analysis and Spark SQL


In [16]:
quality_control_df.createOrReplaceTempView("quality_control")
maintenance_records_df.createOrReplaceTempView("maintenance_records")
machines_df.createOrReplaceTempView("machines")
sensor_data_df.createOrReplaceTempView("sensor_data")

print("DataFrames registered as temporary SQL tables.")

DataFrames registered as temporary SQL tables.


### Analyze Machine Utilization



In [17]:
from pyspark.sql.functions import avg

machine_utilization_query = """
    SELECT
        m.machine_type,
        s.machine_id,
        AVG(s.rpm) AS avg_rpm,
        AVG(s.power_kw) AS avg_power_kw,
        COUNT(s.sensor_id) AS sensor_readings_count
    FROM
        sensor_data s
    JOIN
        machines m ON s.machine_id = m.machine_id
    GROUP BY
        m.machine_type, s.machine_id
    ORDER BY
        m.machine_type, avg_power_kw DESC
"""

machine_utilization_df = spark.sql(machine_utilization_query)

print("Average RPM and Power Consumption per Machine:")
machine_utilization_df.show()

Average RPM and Power Consumption per Machine:
+----------------+----------+------------------+------------------+---------------------+
|    machine_type|machine_id|           avg_rpm|      avg_power_kw|sensor_readings_count|
+----------------+----------+------------------+------------------+---------------------+
|   Conveyor_Belt|      M009| 2397.313333333331| 31.03250000000002|                  540|
|   Conveyor_Belt|      M007|1699.6398148148128| 30.56714814814813|                  540|
|   Conveyor_Belt|      M008|1791.0481481481495|30.547685185185152|                  540|
|   Conveyor_Belt|      M012|2954.2418518518516|  30.1239814814815|                  540|
|   Conveyor_Belt|      M003|          1522.805|29.312314814814822|                  540|
| Hydraulic_Press|      M006| 2087.893888888889|30.449370370370385|                  540|
| Hydraulic_Press|      M019|2044.7988888888892|30.183407407407422|                  540|
| Hydraulic_Press|      M015|1496.2729629629632|29.90

### Determine Production Efficiency



In [19]:
from pyspark.sql.functions import sum, col

production_efficiency_query = """
    SELECT
        m.machine_type,
        qc.machine_id,
        SUM(qc.units_produced) AS total_units_produced,
        SUM(qc.defect_count) AS total_defects,
        (CAST(SUM(qc.defect_count) AS DOUBLE) / SUM(qc.units_produced)) * 100 AS defect_rate_percentage
    FROM
        quality_control qc
    JOIN
        machines m ON qc.machine_id = m.machine_id
    GROUP BY
        m.machine_type, qc.machine_id
    ORDER BY
        defect_rate_percentage DESC
"""

production_efficiency_df = spark.sql(production_efficiency_query)

print("Production Efficiency (Total Units Produced, Total Defects, Defect Rate) per Machine:")
production_efficiency_df.show()

Production Efficiency (Total Units Produced, Total Defects, Defect Rate) per Machine:
+----------------+----------+--------------------+-------------+----------------------+
|    machine_type|machine_id|total_units_produced|total_defects|defect_rate_percentage|
+----------------+----------+--------------------+-------------+----------------------+
|     Robotic_Arm|      M010|                9634|          413|     4.286900560514844|
|Injection_Molder|      M001|               11000|          400|    3.6363636363636362|
|     Robotic_Arm|      M005|               17682|          583|    3.2971383327677866|
| Hydraulic_Press|      M006|               15436|          506|    3.2780513086291783|
| Hydraulic_Press|      M019|                8178|          267|    3.2648569332355097|
|     Robotic_Arm|      M014|               15447|          476|    3.0815044992555185|
|     Robotic_Arm|      M002|               15219|          462|     3.035679085353834|
|Injection_Molder|      M011|     

### Identify Failure Patterns


In [18]:
failure_patterns_query = """
    SELECT
        m.machine_type,
        qc.machine_id,
        COUNT(qc.qc_id) AS total_failed_batches,
        SUM(qc.defect_count) AS total_defects_in_failed_batches
    FROM
        quality_control qc
    JOIN
        machines m ON qc.machine_id = m.machine_id
    WHERE
        qc.pass_fail = 'FAIL'
    GROUP BY
        m.machine_type, qc.machine_id
    ORDER BY
        total_failed_batches DESC, total_defects_in_failed_batches DESC
"""

failure_patterns_df = spark.sql(failure_patterns_query)

print("Failure Patterns by Machine Type and Machine ID:")
failure_patterns_df.show()

Failure Patterns by Machine Type and Machine ID:
+----------------+----------+--------------------+-------------------------------+
|    machine_type|machine_id|total_failed_batches|total_defects_in_failed_batches|
+----------------+----------+--------------------+-------------------------------+
| Hydraulic_Press|      M015|                   1|                             59|
|Injection_Molder|      M020|                   1|                             29|
+----------------+----------+--------------------+-------------------------------+



### Analyze Maintenance Schedules


In [20]:
maintenance_analysis_query = """
    SELECT
        m.machine_type,
        mr.machine_id,
        COUNT(mr.maintenance_id) AS total_maintenance_events,
        AVG(mr.downtime_hours) AS avg_downtime_hours,
        SUM(mr.cost_usd) AS total_maintenance_cost_usd
    FROM
        maintenance_records mr
    JOIN
        machines m ON mr.machine_id = m.machine_id
    GROUP BY
        m.machine_type, mr.machine_id
    ORDER BY
        total_maintenance_events DESC, total_maintenance_cost_usd DESC
"""

maintenance_analysis_df = spark.sql(maintenance_analysis_query)

print("Maintenance Schedule Analysis (Total Events, Avg Downtime, Total Cost) per Machine:")
maintenance_analysis_df.show()

Maintenance Schedule Analysis (Total Events, Avg Downtime, Total Cost) per Machine:
+----------------+----------+------------------------+------------------+--------------------------+
|    machine_type|machine_id|total_maintenance_events|avg_downtime_hours|total_maintenance_cost_usd|
+----------------+----------+------------------------+------------------+--------------------------+
|Injection_Molder|      M011|                       9| 3.725555555555555|                   6843.21|
|     Robotic_Arm|      M002|                       9| 3.572222222222222|                   6757.52|
|   Welding_Robot|      M018|                       9|3.5266666666666664|                   6276.49|
| Hydraulic_Press|      M006|                       9| 2.907777777777778|         5994.530000000001|
|   Conveyor_Belt|      M003|                       9|2.6966666666666668|         5562.999999999999|
|   Welding_Robot|      M013|                       8|3.1325000000000003|                   5983.52|
|     R

### Generate Monthly Production Reports



In [21]:
from pyspark.sql.functions import year, month, sum, col

monthly_production_query = """
    SELECT
        YEAR(inspection_date) AS production_year,
        MONTH(inspection_date) AS production_month,
        SUM(units_produced) AS total_monthly_units_produced,
        SUM(defect_count) AS total_monthly_defects,
        (CAST(SUM(defect_count) AS DOUBLE) / SUM(units_produced)) * 100 AS average_monthly_defect_rate_percentage
    FROM
        quality_control
    GROUP BY
        production_year, production_month
    ORDER BY
        production_year, production_month
"""

monthly_production_df = spark.sql(monthly_production_query)

print("Monthly Production Report:")
monthly_production_df.show()

Monthly Production Report:
+---------------+----------------+----------------------------+---------------------+--------------------------------------+
|production_year|production_month|total_monthly_units_produced|total_monthly_defects|average_monthly_defect_rate_percentage|
+---------------+----------------+----------------------------+---------------------+--------------------------------------+
|           2025|               1|                      109581|                 3249|                    2.9649300517425465|
|           2025|               2|                       97516|                 2846|                    2.9184954263915666|
|           2025|               3|                       94056|                 2694|                    2.8642510844603217|
+---------------+----------------+----------------------------+---------------------+--------------------------------------+



## Q6. ETL Pipeline Development



### ETL Pipeline: Integrating Machine and Maintenance Data

The pipeline will involve:
1.  **Extract:** Using the already loaded `machines_df` and `maintenance_records_df`.
2.  **Transform:** Joining these DataFrames and aggregating maintenance data to calculate the total number of maintenance events, average downtime, and total cost per machine.
3.  **Load:** Displaying the transformed DataFrame.

In [22]:
from pyspark.sql.functions import count, avg, sum

# 1. Extract (DataFrames are already loaded: machines_df, maintenance_records_df)

# 2. Transform: Join and Aggregate
maintenance_etl_df = maintenance_records_df.join(
    machines_df,
    on='machine_id',
    how='inner'
).groupBy("machine_id", "machine_type", "location").agg(
    count("maintenance_id").alias("total_maintenance_events"),
    avg("downtime_hours").alias("average_downtime_hours"),
    sum("cost_usd").alias("total_maintenance_cost_usd")
).orderBy("total_maintenance_events", ascending=False)

# 3. Load: Display the transformed DataFrame
print("ETL Pipeline Result: Integrated Maintenance and Machine Data (first 10 rows):")
maintenance_etl_df.show(10)

ETL Pipeline Result: Integrated Maintenance and Machine Data (first 10 rows):
+----------+----------------+--------------+------------------------+----------------------+--------------------------+
|machine_id|    machine_type|      location|total_maintenance_events|average_downtime_hours|total_maintenance_cost_usd|
+----------+----------------+--------------+------------------------+----------------------+--------------------------+
|      M011|Injection_Molder|Plant_B_Floor2|                       9|     3.725555555555555|                   6843.21|
|      M018|   Welding_Robot|Plant_B_Floor2|                       9|    3.5266666666666664|                   6276.49|
|      M003|   Conveyor_Belt|Plant_A_Floor2|                       9|    2.6966666666666668|         5562.999999999999|
|      M002|     Robotic_Arm|Plant_A_Floor1|                       9|     3.572222222222222|                   6757.52|
|      M006| Hydraulic_Press|Plant_B_Floor2|                       9|     2.907777

## Q7. Machine Learning/Deep Learning Implementation



1.  **Feature Engineering:**
2.  **Model Training:**
3.  **Model Evaluation:**

In [23]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Join quality_control_df with machines_df to get machine_type and capacity
ml_data_df = quality_control_df.join(machines_df, on='machine_id', how='inner')

# Select relevant columns for the model
# We'll predict 'defect_count' using 'units_produced', 'machine_type', and 'capacity_units_per_hour'
selected_data = ml_data_df.select("units_produced", "machine_type", "capacity_units_per_hour", "defect_count")

# Handle categorical feature 'machine_type' using StringIndexer and OneHotEncoder
stringIndexer = StringIndexer(inputCol="machine_type", outputCol="machine_type_index")
oneHotEncoder = OneHotEncoder(inputCol="machine_type_index", outputCol="machine_type_vec")

# Assemble features into a single vector column
assembler = VectorAssembler(
    inputCols=["units_produced", "capacity_units_per_hour", "machine_type_vec"],
    outputCol="features"
)

# Create a pipeline to apply transformations
pipeline = Pipeline(stages=[stringIndexer, oneHotEncoder, assembler])

# Fit the pipeline to the data and transform it
model_data = pipeline.fit(selected_data).transform(selected_data)

# Split the data into training and test sets
(trainingData, testData) = model_data.randomSplit([0.7, 0.3], seed=42)

print("Data prepared for ML. First 5 rows of training data (features and label):")
trainingData.select("features", "defect_count").show(5, truncate=False)

Data prepared for ML. First 5 rows of training data (features and label):
+-----------------------------+------------+
|features                     |defect_count|
+-----------------------------+------------+
|[104.0,264.0,0.0,0.0,0.0,1.0]|3           |
|[106.0,155.0,0.0,1.0,0.0,0.0]|4           |
|[109.0,211.0,1.0,0.0,0.0,0.0]|6           |
|[110.0,439.0,1.0,0.0,0.0,0.0]|1           |
|[110.0,451.0,1.0,0.0,0.0,0.0]|5           |
+-----------------------------+------------+
only showing top 5 rows


### Model Training and Evaluation : Linear Regression



In [24]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize Linear Regression model
lr = LinearRegression(featuresCol="features", labelCol="defect_count")

# Train the model
lr_model = lr.fit(trainingData)

# Make predictions on the test data
predictions = lr_model.transform(testData)

# Show predictions (actual vs. predicted defect_count)
print("Predictions vs. Actual Defect Count (first 5 rows):")
predictions.select("defect_count", "prediction").show(5)

# Evaluate the model
evaluator_rmse = RegressionEvaluator(labelCol="defect_count", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="defect_count", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE) on test data = {rmse}")
print(f"R-squared (R2) on test data = {r2}")

Predictions vs. Actual Defect Count (first 5 rows):
+------------+------------------+
|defect_count|        prediction|
+------------+------------------+
|           0|3.7645151379571113|
|           5| 3.663768842775515|
|           3|1.8932455044442116|
|           2|3.3811490491617087|
|           4|1.6248559179378068|
+------------+------------------+
only showing top 5 rows
Root Mean Squared Error (RMSE) on test data = 9.973795272191161
R-squared (R2) on test data = 0.36305712518909883
